In [ ]:
# ==========================================
# FABRIC BRONZE LAYER HEALTH ASSESSMENT
# ==========================================
# This notebook assesses the structural integrity, freshness, 
# completeness, and historical volume stability of Bronze tables.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import datetime

# --- CONFIGURATION ---
CONFIG = {
    "expected_tables": ["customers_bronze", "sales_bronze", "products_bronze"],
    "expected_columns": {
        "customers_bronze": ["CustomerID", "CustomerName", "IngestionTimestamp"],
        "sales_bronze": ["SalesID", "CustomerID", "SaleAmount", "IngestionTimestamp"],
        "products_bronze": ["ProductID", "ProductName", "IngestionTimestamp"]
    },
    "max_acceptable_null_pct": 5.0,
    "volume_anomaly_threshold_pct": 50.0,
    "history_table_name": "bronze_health_history"
}

current_run_id = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
assessment_timestamp = datetime.datetime.utcnow()

print(f"Starting Bronze Health Assessment | Run ID: {current_run_id}")

In [ ]:
# --- STEP 1: TABLE DISCOVERY & TECHNICAL HEALTH ---
spark = SparkSession.builder.getOrCreate()
catalog_tables = [t.name for t in spark.catalog.listTables()]

assessment_results = []

for table_name in CONFIG["expected_tables"]:
    table_exists = table_name in catalog_tables
    
    if not table_exists:
        assessment_results.append({
            "RunID": current_run_id,
            "AssessmentTimestamp": assessment_timestamp,
            "TableName": table_name,
            "CheckCategory": "Structural",
            "CheckName": "Table Existence",
            "Status": "FAIL",
            "Details": f"Table '{table_name}' is missing from the Lakehouse."
        })
        continue

    df = spark.sql(f"SELECT * FROM {table_name}")
    row_count = df.count()
    column_count = len(df.columns)
    
    assessment_results.append({
        "RunID": current_run_id,
        "AssessmentTimestamp": assessment_timestamp,
        "TableName": table_name,
        "CheckCategory": "Structural",
        "CheckName": "Table Existence",
        "Status": "PASS",
        "Details": f"Table exists with {row_count} rows and {column_count} columns."
    })
    
    missing_cols = [c for c in CONFIG["expected_columns"].get(table_name, []) if c not in df.columns]
    col_status = "FAIL" if missing_cols else "PASS"
    assessment_results.append({
        "RunID": current_run_id,
        "AssessmentTimestamp": assessment_timestamp,
        "TableName": table_name,
        "CheckCategory": "Structural",
        "CheckName": "Required Columns",
        "Status": col_status,
        "Details": f"Missing columns: {missing_cols}" if missing_cols else "All expected columns present."
    })

In [ ]:
# --- STEP 2: INGESTION FRESHNESS & COMPLETENESS ---
for table_name in CONFIG["expected_tables"]:
    if table_name not in catalog_tables:
        continue
        
    df = spark.sql(f"SELECT * FROM {table_name}")
    
    if "IngestionTimestamp" in df.columns:
        latest_ingestion = df.agg(F.max("IngestionTimestamp")).collect()[0][0]
        
        if latest_ingestion is None:
            freshness_status = "WARNING"
            freshness_details = "IngestionTimestamp column contains nulls or no records."
        else:
            freshness_status = "PASS"
            freshness_details = f"Latest ingestion recorded at: {latest_ingestion}"
            
        assessment_results.append({
            "RunID": current_run_id,
            "AssessmentTimestamp": assessment_timestamp,
            "TableName": table_name,
            "CheckCategory": "Ingestion",
            "CheckName": "Data Freshness",
            "Status": freshness_status,
            "Details": freshness_details
        })

    total_rows = df.count()
    if total_rows > 0:
        for col in CONFIG["expected_columns"].get(table_name, []):
            if col in df.columns:
                null_count = df.filter(F.col(col).isNull()).count()
                null_pct = (null_count / total_rows) * 100
                status = "FAIL" if null_pct > CONFIG["max_acceptable_null_pct"] else "PASS"
                
                assessment_results.append({
                    "RunID": current_run_id,
                    "AssessmentTimestamp": assessment_timestamp,
                    "TableName": table_name,
                    "CheckCategory": "Completeness",
                    "CheckName": f"Null Check: {col}",
                    "Status": status,
                    "Details": f"Null percentage: {null_pct:.2f}% (Threshold: {CONFIG['max_acceptable_null_pct']}%)"
                })

In [ ]:
# --- STEP 3: HISTORICAL SNAPSHOT COMPARISON & SCORECARD ---
import pandas as pd

results_df = pd.DataFrame(assessment_results)
display(results_df)

spark_results_df = spark.createDataFrame(results_df)
spark_results_df.write.format("delta").mode("append").saveAsTable(CONFIG["history_table_name"])
print("Health assessment results successfully logged to historical Delta table.")